# GEE head (RepViT-M2.3) on LoveDA — Kaggle T4 x2

Before running: turn on **Internet**, set the accelerator to **GPU T4 x2**, and attach a LoveDA dataset with the layout
`Train|Val|Test / Urban|Rural / images_png, masks_png`. The code is cloned from GitHub in the first cell.

Nothing here has been run on Kaggle yet: expect to fix small things (paths, package versions) on the first pass.

In [ ]:
import os, shutil, subprocess, sys, time

REPO_URL = "https://github.com/TuanHungGiang/LandCover"
DATA_INPUT = "/kaggle/input"          # searched for Train/Urban/images_png
WORK = "/kaggle/working/LandCover"

def find_dir(base, test):
    for root, dirs, files in os.walk(base):
        if test(root, dirs, files):
            return root
    raise FileNotFoundError(f"nothing under {base} matched")

if os.path.isdir(WORK):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)     # re-running the notebook: update the code
else:
    subprocess.run(["git", "clone", REPO_URL, WORK], check=True)

data_root = find_dir(DATA_INPUT, lambda r, d, f: os.path.isdir(os.path.join(r, "Train", "Urban", "images_png")))
print("data :", data_root)

os.makedirs(f"{WORK}/data", exist_ok=True)
link = f"{WORK}/data/2021LoveDA"
if not os.path.exists(link):
    os.symlink(data_root, link)
os.chdir(WORK)
print(sorted(os.listdir(link)), os.getcwd())

In [ ]:
!pip -q install pytorch-lightning==2.0.6 torchmetrics==0.11.4 timm==0.6.7 einops addict prettytable ttach safetensors

In [ ]:
# import checks (repo code needs SqueezeExcite / trunc_normal_ from this timm version)
!python -c "import torch, pytorch_lightning as pl, torchmetrics, timm; from timm.models.layers import SqueezeExcite; from timm.models.vision_transformer import trunc_normal_; print(torch.__version__, torch.cuda.device_count(), 'GPUs |', pl.__version__, torchmetrics.__version__, timm.__version__)"

## RepViT-M2.3 ImageNet weights (converted from the timm release on Hugging Face)

In [ ]:
!python tools/convert_repvit_timm.py --dst pretrain/repvit_m2_3_distill_450e.pth

## Timing / memory run (GPU 0)
40 training batches + 8 validation batches, fp32 then fp16-mixed. Look at the peak memory and the seconds per batch to decide the real epoch count
(the full training set is 2522 images = 1261 batches of 2). fp16 has not been tested with this head: if it fails, use fp32.

In [ ]:
def write_cfg(name, precision):
    os.makedirs(f"work_dirs/{name}", exist_ok=True)
    open(f"configs/loveda/{name}.py", "w").write(f'''
_base_ = './gee_repvit.py'
exp_name = "work_dirs/{name}"
epoch = 1
precision = "{precision}"
limit_train_batches = 40
limit_val_batches = 8
optimizer_config = dict(scheduler=dict(max_epoch=1))
''')

def gpu_mem(gpu=0):
    out = subprocess.run(["nvidia-smi", f"--id={gpu}", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
    return int(out.strip().splitlines()[0])

def run_timed(cfg, gpu=0):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    t0, peak = time.time(), 0
    p = subprocess.Popen([sys.executable, "train.py", "-c", cfg], stdout=open(cfg.split("/")[-1] + ".log", "w"), stderr=subprocess.STDOUT, env=env)
    while p.poll() is None:
        peak = max(peak, gpu_mem(gpu)); time.sleep(3)
    return p.returncode, time.time() - t0, peak

for name, prec in [("quick_fp32", "32"), ("quick_fp16", "16-mixed")]:
    write_cfg(name, prec)
    rc, secs, peak = run_timed(f"configs/loveda/{name}.py")
    print(f"{name}: exit {rc} | {secs:.0f}s total (includes model build + validation) | peak GPU memory {peak} MiB")
    if rc != 0:
        print(open(f"{name}.py.log").read()[-3000:])

## Full runs, both GPUs in parallel
* GPU 0: `gee_repvit.py` (gated exploit/explore)
* GPU 1: `gee_repvit_exploit.py` (same code with the explore path and gate removed = ablation baseline)

Each process only sees its own GPU (`CUDA_VISIBLE_DEVICES`), so the configs stay `gpus=[0]`. Check the timing cell first: 50 epochs may not fit in a 12 h session.
To change the number of epochs or the precision, add lines to a config (`epoch = 30`, the scheduler `max_epoch`, `precision = "16-mixed"`).
Kaggle has ~4 CPU cores: if the GPUs wait for data, lower `num_workers` in the config.

In [ ]:
CONFIGS = {0: "configs/loveda/gee_repvit.py", 1: "configs/loveda/gee_repvit_exploit.py"}
procs = {}
for gpu, cfg in CONFIGS.items():
    exp = {"gee_repvit.py": "gee_repvit_loveda", "gee_repvit_exploit.py": "gee_repvit_exploit_loveda"}[cfg.split("/")[-1]]
    os.makedirs(f"work_dirs/{exp}", exist_ok=True)   # train.py appends to work_dirs/<exp>/train_metric.txt
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    procs[gpu] = subprocess.Popen([sys.executable, "train.py", "-c", cfg], stdout=open(f"full_gpu{gpu}.log", "w"), stderr=subprocess.STDOUT, env=env)
    print("started", cfg, "on GPU", gpu)

In [ ]:
# progress (re-run this cell any time)
for gpu, p in procs.items():
    print(f"--- GPU {gpu}: {'running' if p.poll() is None else 'exit ' + str(p.returncode)}")
    print(open(f"full_gpu{gpu}.log").read()[-800:])

In [ ]:
# wait for both runs to finish, then show the last validation table of each
while any(p.poll() is None for p in procs.values()):
    time.sleep(60)
for exp in ["gee_repvit_loveda", "gee_repvit_exploit_loveda"]:
    txt = open(f"work_dirs/{exp}/train_metric.txt").read()
    print("=====", exp); print(txt[-1800:])
print(subprocess.run("ls -la work_dirs/*/*.ckpt", shell=True, capture_output=True, text=True).stdout)